# Gemma 4 31B V6.1 Hybrid + 3-Bit Vision + Live CRTR/MoD (Colab Pro A100)

**Architecture Upgrade over V6.0 (`v5-qat-dev`):**
- `L00..L35`: V6.0 LittleBit-2 1-bit Row-Major (`txt_cos 0.966..0.992`).
- `L36..L47`: 2-bit Gauss Row-Major (`lr_rank=16`) Attention + LittleBit-2 (`s=2560, sr=1024`) MLP.
- `L48..L55`: 2-bit Gauss Row-Major (`lr_rank=16`) Attention + MLP.
- `L56..L59`: Joint 4-Layer Logit-KL Distilled 2-bit Gauss Row-Major (`lr_rank=32`) Tail.
- **Vision Tower (`V00..V26`):** Full 3-bit (`3b_u0 @ b64`, `~0.234 GB`).
- **Live CRTR + KV-Safe MoD:** Representation-matched `gemma4_31b_v6_1_router.crtr` (`CRTR_FLAG_MOD_GATE`, `~10.86 MB`) trained on `inp` (`input_layernorm`) & `hm` (`pre_feedforward_layernorm`) directly on V6.1 student activations.
- **Pi 4 Footprint:** `~4.68 GB` (`~4.36 GiB <= 4.95 GB` hard guard) + zero-hole `.partial`-first Drive checkpointing.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DannyB-bit/Language-U-Gemma4.git"
REPO_DIR = Path("/content/Language-U-Gemma4")
DRIVE_V61 = Path("/content/drive/MyDrive/gemma4_v6_1_hybrid")
DRIVE_V61.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-b", "v5-qat-dev", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "v5-qat-dev"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/v5-qat-dev"], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=True)
subprocess.run(["nvidia-smi"], check=True)

# Auto-check required dependencies and install latest compatible wheels if missing
_need_pip = False
try:
    import datasets, safetensors, tokenizers, huggingface_hub, PIL
    from transformers.models.gemma4.modeling_gemma4 import Gemma4VisionModel
except Exception:
    _need_pip = True
if _need_pip:
    print("[Setup] Installing transformers>=4.51.0 accelerate safetensors huggingface_hub datasets tokenizers pillow...")
    import sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "transformers>=4.51.0", "accelerate", "safetensors", "huggingface_hub", "datasets", "tokenizers", "pillow"],
        check=True,
    )
from transformers.models.gemma4.modeling_gemma4 import Gemma4VisionModel  # verify Gemma 4 class
import transformers
print(f"[Setup OK] transformers={transformers.__version__}, Gemma4VisionModel importable")

# Optional Hugging Face token from Colab Secrets (if needed for gated weights)
try:
    from google.colab import userdata
    hf_tok = userdata.get("HF_TOKEN")
    if hf_tok:
        os.environ["HF_TOKEN"] = hf_tok
        print("Loaded HF_TOKEN from Colab secrets.")
except Exception:
    pass

In [ ]:
import os
import subprocess
import time
from pathlib import Path

SCRIPT = Path("/content/Language-U-Gemma4/experiments/run270_rust/calib/31b/run_v6_1_hybrid_crtr_gpu.py")
LOG_PATH = Path("/content/v6_1_train.log")

env = os.environ.copy()
env["V6_1_DRIVE_DIR"] = "/content/drive/MyDrive/gemma4_v6_1_hybrid"
env["V6_1_OUT_DIR"] = "/content/v6_1_outputs"
env["COLAB_AUTO_UNASSIGN"] = "1"
env["PYTHONUNBUFFERED"] = "1"

print("Running 6-gate V6.1 Hybrid + CRTR pre-flight selftest...")
subprocess.run(["python3", str(SCRIPT), "--selftest"], env=env, check=True)

print(f"Launching A100 calibration -> {LOG_PATH} (auto-unassign enabled)...")
with open(LOG_PATH, "w", encoding="utf-8") as lfp:
    proc = subprocess.Popen(["python3", "-u", str(SCRIPT)], stdout=lfp, stderr=subprocess.STDOUT, env=env)

print(f"Started PID={proc.pid}; streaming log live...")
seen = 0
while proc.poll() is None:
    time.sleep(15)
    if LOG_PATH.is_file():
        txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
        if len(txt) > seen:
            print(txt[seen:], end="", flush=True)
            seen = len(txt)

txt = LOG_PATH.read_text(encoding="utf-8", errors="replace")
if len(txt) > seen:
    print(txt[seen:], end="", flush=True)
assert proc.returncode == 0, f"V6.1 pipeline exited with code {proc.returncode}"

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

sys.path.insert(0, "/content/Language-U-Gemma4/experiments/run270_rust/calib/31b")
import run_v6_1_hybrid_crtr_gpu as v61
import train_cortical_router as crtr

DRIVE_V61 = Path("/content/drive/MyDrive/gemma4_v6_1_hybrid")
t16c_p = DRIVE_V61 / "gemma4_31b_v6_1_hybrid.t16c"
crtr_p = DRIVE_V61 / "gemma4_31b_v6_1_router.crtr"
rpt_p = DRIVE_V61 / "v6_1_hybrid_qat_report.json"

for fp in sorted(DRIVE_V61.iterdir()):
    if fp.is_file():
        h = hashlib.sha256(fp.read_bytes()).hexdigest()[:16] if fp.stat().st_size < 100_000_000 else v61.sha256_file(fp)[:16]
        print(f"{fp.name:<38s} {fp.stat().st_size:>14,d} B  sha256={h}...")

if t16c_p.is_file():
    nm, nb = v61.verify_t16c_integrity(t16c_p)
    print(f"[VERIFIED T16C] mats={nm}, eof={nb:,} B ({nb/1024**3:.3f} GiB)")
if crtr_p.is_file():
    cmeta, clayers = crtr.load_crtr(crtr_p)
    print(f"[VERIFIED CRTR] layers={len(clayers)} d={cmeta['d_model']} ff={cmeta['ff']} rank={cmeta['rank']} flags=0x{cmeta['flags']:x}")
if rpt_p.is_file():
    print(json.dumps(json.loads(rpt_p.read_text(encoding="utf-8")), indent=2)[:2000])